# RG3 가스 최종 모델 내보내기
첫 코드 셀은 순수 정의. 두 번째 코드 셀만 학습/저장한다. 미성형 원본 대기 중.

In [ ]:
from pathlib import Path
import json, hashlib, sys, joblib, numpy as np, pandas as pd, sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
RAW=['Injection_Time','Filling_Time','Plasticizing_Time','Cycle_Time','Clamp_Close_Time','Cushion_Position','Plasticizing_Position','Clamp_Open_Position','Max_Injection_Speed','Max_Screw_RPM','Average_Screw_RPM','Max_Injection_Pressure','Max_Switch_Over_Pressure','Max_Back_Pressure','Average_Back_Pressure']+['Barrel_Temperature_'+str(k) for k in range(1,7)]+['Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4']
KEYS=['EQUIP_CD','product','TimeStamp']
STD=[v+'__past10_std' for v in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
REL=[('Max_Injection_Pressure',['Max_Injection_Speed','Filling_Time']),('Plasticizing_Time',['Average_Screw_RPM','Average_Back_Pressure']),('Cushion_Position',['Plasticizing_Position','Max_Injection_Pressure']),('Mold_Temperature_4',['Mold_Temperature_3','Barrel_Temperature_5'])]
NEEDED=list(dict.fromkeys(RAW+STD+[v+s for v,s in [
 ('Barrel_Temperature_5','__delta1'),('Mold_Temperature_3','__delta1'),('Mold_Temperature_4','__delta1'),('Mold_Temperature_3','__dev10'),('Mold_Temperature_4','__dev10'),('Max_Injection_Pressure','__delta1'),('Max_Switch_Over_Pressure','__delta1'),('Max_Injection_Speed','__delta1'),('Filling_Time','__delta1'),('Cushion_Position','__delta1'),('Max_Injection_Pressure','__dev10'),('Max_Switch_Over_Pressure','__dev10')]]))
def normalize_raw(frame):
 z=frame.copy()
 if 'product' not in z and 'PART_NAME' in z:z['product']=z.PART_NAME.str.extract('(CN7|RG3)',expand=False)
 missing=set(KEYS+RAW)-set(z)
 if missing:raise ValueError('Missing raw columns: '+str(sorted(missing)))
 z=z[KEYS+RAW].copy();z.TimeStamp=pd.to_datetime(z.TimeStamp,errors='raise')
 if z[KEYS].isna().any().any():raise ValueError('Missing equipment/product/Timestamp')
 z[RAW]=z[RAW].apply(pd.to_numeric,errors='raise')
 z[RAW]=z[RAW].replace([np.inf,-np.inf],np.nan)
 return z
def prepare_features(query_raw,context_raw=None):
 q=normalize_raw(query_raw); context=normalize_raw(context_raw) if context_raw is not None else q.iloc[:0]
 qkeys=pd.MultiIndex.from_frame(q[KEYS]).unique()
 # Query sensors win identical keys; conflicts/ambiguous product remain abstentions.
 allrows=pd.concat([q,context],ignore_index=True)
 ranges=allrows.groupby(KEYS)[RAW].max()-allrows.groupby(KEYS)[RAW].min()
 conflicts=ranges.gt(1e-4).any(axis=1).rename('sensor_conflict')
 obs=allrows.drop_duplicates(KEYS).merge(conflicts,on=KEYS,how='left').sort_values(['EQUIP_CD','TimeStamp','product']).reset_index(drop=True)
 ambiguous=obs.groupby(['EQUIP_CD','TimeStamp'])['product'].transform('nunique').gt(1)
 obs['sensor_conflict']=obs.sensor_conflict|ambiguous
 # Match old builder's removal of ambiguous history keys.
 ambiguous_keys=pd.MultiIndex.from_frame(obs.loc[ambiguous,KEYS])
 invalid_query=q.drop_duplicates(KEYS)[pd.MultiIndex.from_frame(q.drop_duplicates(KEYS)[KEYS]).isin(ambiguous_keys)].copy()
 obs=obs[~ambiguous].copy().reset_index(drop=True)
 obs.loc[obs.sensor_conflict,RAW]=np.nan
 gap=obs.groupby('EQUIP_CD').TimeStamp.diff().dt.total_seconds()
 switch=obs['product'].ne(obs.groupby('EQUIP_CD')['product'].shift())
 obs['history_segment']=(switch|gap.gt(1800)|gap.isna()).cumsum()
 for v in RAW:
  prev=obs.groupby('history_segment')[v].shift()
  grouped=prev.groupby(obs.history_segment)
  obs[v+'__delta1']=obs[v]-prev
  obs[v+'__past10_std']=grouped.transform(lambda x:x.rolling(10,min_periods=5).std())
  obs[v+'__dev10']=obs[v]-grouped.transform(lambda x:x.rolling(10,min_periods=5).median())
 selected=obs[pd.MultiIndex.from_frame(obs[KEYS]).isin(qkeys)].copy()
 if len(invalid_query):
  invalid_query['sensor_conflict']=True
  selected=pd.concat([selected,invalid_query],ignore_index=True)
 return selected.sort_values(KEYS).reset_index(drop=True)
def fit_relations(normal):
 result=[]
 for target,parents in REL:
  sc=RobustScaler().fit(normal[parents]);m=Ridge(alpha=10).fit(sc.transform(normal[parents]),normal[target])
  error=normal[target].to_numpy()-m.predict(sc.transform(normal[parents]))
  scale=max(float(np.sqrt(np.mean(error**2))),float(normal[target].std(ddof=0))*.1,1e-6)
  result.append(dict(target=target,parents=parents,scaler=sc,model=m,scale=scale))
 return result
def relation_features(models,q):
 return np.column_stack([(q[r['target']].to_numpy()-r['model'].predict(r['scaler'].transform(q[r['parents']]))) / r['scale'] for r in models])
def physical(q,name):
 mold=(q.Mold_Temperature_3+q.Mold_Temperature_4)/2
 gap=q.Barrel_Temperature_5__delta1-(q.Mold_Temperature_3__delta1+q.Mold_Temperature_4__delta1)/2
 if name=='thermal':
  return np.column_stack([q[['Barrel_Temperature_'+str(k) for k in range(1,7)]].mean(axis=1)-mold,q.Mold_Temperature_3-q.Mold_Temperature_4,(q.Mold_Temperature_3__delta1+q.Mold_Temperature_4__delta1)/2,(q.Mold_Temperature_3__dev10+q.Mold_Temperature_4__dev10)/2,gap,q.Max_Injection_Pressure/q.Filling_Time.where(q.Filling_Time.abs()>1e-6)])
 return np.column_stack([q.Max_Injection_Pressure__delta1-q.Max_Switch_Over_Pressure__delta1,q.Max_Injection_Speed__delta1,q.Filling_Time__delta1,q.Max_Injection_Pressure__dev10-q.Max_Switch_Over_Pressure__dev10,q.Cushion_Position__delta1,gap])
def conditional_features(peer,q):
 raw=peer['raw_imp'].transform(physical(q,peer['name']))
 control=peer['control_sc'].transform(peer['control_imp'].transform(q[peer['controls']]))
 neighbors=peer['nn'].kneighbors(control,return_distance=False)
 out=[]
 for j,ids in enumerate(neighbors):
  # Exclude the same normal Shot when constructing training local residuals.
  same=(peer['keys'].iloc[ids].to_numpy()==q[KEYS].iloc[j].to_numpy()).all(axis=1)
  ids=ids[~same][:30]
  if not len(ids):ids=np.arange(len(peer['raw']))
  out.append((raw[j]-np.median(peer['raw'][ids],axis=0))/np.maximum(np.std(peer['raw'][ids],axis=0),peer['floor']))
 return np.asarray(out)
def fit_conditional(d,y,name,seeds):
 normal=d.loc[y==0]
 controls=['Barrel_Temperature_5','Max_Injection_Speed','Filling_Time','Hopper_Temperature'] if name=='thermal' else ['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3']
 ri=SimpleImputer(strategy='median',keep_empty_features=True).fit(physical(normal,name))
 ci=SimpleImputer(strategy='median',keep_empty_features=True).fit(normal[controls]);cs=RobustScaler().fit(ci.transform(normal[controls]))
 raw=ri.transform(physical(normal,name))
 peer=dict(name=name,controls=controls,raw_imp=ri,control_imp=ci,control_sc=cs,raw=raw,keys=normal[KEYS].reset_index(drop=True),floor=np.maximum(np.std(raw,axis=0)*.1,1e-6),nn=NearestNeighbors(n_neighbors=min(31,len(normal))).fit(cs.transform(ci.transform(normal[controls]))))
 z=conditional_features(peer,d);imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(z);z=imp.transform(z)
 trees=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  trees.append([cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=s,n_jobs=2).fit(z,y) for s in seeds])
 sc=RobustScaler().fit(z[y==0]);lr=[LogisticRegression(C=c,class_weight='balanced',max_iter=2000,random_state=42).fit(sc.transform(z),y) for c in [.1,1.,10.]]
 return dict(peer=peer,imputer=imp,trees=trees,scaler=sc,linear=lr)
def conditional_score(component,q):
 z0=conditional_features(component['peer'],q);z=component['imputer'].transform(z0)
 a,b=[np.mean([m.predict_proba(z)[:,1] for m in group],axis=0) for group in component['trees']]
 c=np.mean([m.predict_proba(component['scaler'].transform(z))[:,1] for m in component['linear']],axis=0)
 return (a+b+c)/3,z0
def fit_gas_bundle(d,cuts):
 d=d.reset_index(drop=True);y=d.PassOrFail.to_numpy(int);normal=y==0
 assert len(d)==583 and y.sum()==17 and normal.sum()==566 and d.TimeStamp.is_unique
 residual=np.full((len(d),4),np.nan)
 for block in d.comparison_block.unique():
  q=d.comparison_block.eq(block).to_numpy()
  residual[q]=relation_features(fit_relations(d.loc[~q & normal]),d.loc[q])
 models=[]
 for x in [d[STD],residual]:
  m=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=2)).fit(x,y)
  models.append(m)
 experts=[]
 for x in [d[STD].to_numpy(float),residual]:
  imp=SimpleImputer(strategy='median',keep_empty_features=True);x=imp.fit_transform(x);sc=RobustScaler().fit(x[normal]);z=sc.transform(x)
  positive=np.flatnonzero(~normal);k=min(2,len(np.unique(z[~normal],axis=0)))
  modes=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(z[~normal]) if k>1 else np.zeros(len(positive),int)
  byseed=[]
  for seed in [42,7,101]:
   group=[]
   for mode in np.unique(modes):
    ids=np.r_[np.flatnonzero(normal),positive[modes==mode]]
    group.append(RandomForestClassifier(n_estimators=100,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=seed,n_jobs=2).fit(x[ids],y[ids]))
   byseed.append(group)
  experts.append(dict(imputer=imp,models=byseed))
 bundle=dict(format='RG3_GAS_FINAL_FIT_V1',raw_columns=RAW,required_features=NEEDED,keys=KEYS,equipment=d.EQUIP_CD.unique().tolist(),cuts=cuts,baseline=models,relations=fit_relations(d.loc[normal]),experts=experts,sklearn_version=sklearn.__version__,train_normal=566,train_gas=17,shortshot_available=False)
 bundle['thermal']=fit_conditional(d,y,'thermal',[42,7,101]);print('Thermal exported',flush=True)
 bundle['pressure']=[]
 for seeds in [[42,7,101],[11,23,37],[13,29,53]]:
  bundle['pressure'].append(fit_conditional(d,y,'pressure',seeds));print('Pressure exported',seeds,flush=True)
 return bundle
def predict_gas_features(bundle,features):
 missing=set(KEYS+NEEDED)-set(features)
 if missing:raise ValueError('Missing feature columns: '+str(sorted(missing)))
 f=features.copy().reset_index(drop=True);f.TimeStamp=pd.to_datetime(f.TimeStamp)
 if f[KEYS].duplicated().any():raise ValueError('Duplicate Shot keys')
 ready=np.isfinite(f[NEEDED].to_numpy(float)).all(axis=1)&f.EQUIP_CD.isin(bundle['equipment'])&f['product'].eq('RG3')
 if 'sensor_conflict' in f:ready &= ~f.sensor_conflict.fillna(True)
 result=f[KEYS].copy();result['status']=np.where(ready,'ok','판정보류');result['gas_prediction']=np.nan
 for col in ['baseline','innovation','thermal','pressure_votes','baseline_score','expert_score','thermal_score']:result[col]=np.nan
 q=f.loc[ready].reset_index(drop=True)
 if not len(q):return result
 cuts=bundle['cuts'];rr=relation_features(bundle['relations'],q)
 bp=np.maximum(bundle['baseline'][0].predict_proba(q[STD])[:,1],bundle['baseline'][1].predict_proba(rr)[:,1])
 view_scores=[]
 for x,view in zip([q[STD].to_numpy(float),rr],bundle['experts']):
  x=view['imputer'].transform(x)
  view_scores.append(np.column_stack([np.mean([m.predict_proba(x)[:,1] for m in group],axis=0) for group in view['models']]))
 ex=np.maximum(view_scores[0],view_scores[1]).mean(axis=1)
 b=bp>cuts['baseline'];e=(ex>cuts['expert'])&(q.Barrel_Temperature_5__delta1.to_numpy()>cuts['barrel5_delta'])
 ts,tz=conditional_score(bundle['thermal'],q);th=(ts>cuts['thermal_score'])&(tz[:,4]<cuts['thermal_gate'])
 direction=(q.Max_Injection_Pressure__delta1.round(2)<0)&(q.Max_Injection_Speed__delta1.round(2)>0)&(q.Max_Switch_Over_Pressure__delta1.round(2)<q.Max_Injection_Pressure__delta1.round(2))
 votes=[]
 for component,tag in zip(bundle['pressure'],['42_7_101','11_23_37','13_29_53']):
  score,_=conditional_score(component,q);votes.append((score>cuts['pressure_'+tag])&direction.to_numpy())
 vote=np.column_stack(votes).sum(axis=1)
 values=np.column_stack([(b|e|th|(vote>=2)).astype(int),b,e,th,vote,bp,ex,ts])
 result.loc[ready,['gas_prediction','baseline','innovation','thermal','pressure_votes','baseline_score','expert_score','thermal_score']]=values
 return result
def predict_cascade(bundle,features,shortshot_predictor=None):
 result=predict_gas_features(bundle,features)
 result['predicted_type']=np.where(result.status.ne('ok'),'판정보류',np.where(result.gas_prediction.eq(1),'가스 후보','미성형 모델 미연결'))
 remaining=result.status.eq('ok')&result.gas_prediction.eq(0)
 if shortshot_predictor is not None and remaining.any():
  s=np.asarray(shortshot_predictor(features.loc[remaining].copy()))
  if len(s)!=remaining.sum() or not np.isin(s,[0,1]).all():raise ValueError('Invalid shortshot predictor output')
  result.loc[remaining,'predicted_type']=np.where(s==1,'미성형 후보','정상 예측')
 return result
def save_bundle(bundle,path):
 path=Path(path);path.parent.mkdir(exist_ok=True,parents=True);joblib.dump(bundle,path,compress=3)
 meta=dict(format=bundle['format'],sha256=hashlib.sha256(path.read_bytes()).hexdigest(),sklearn_version=sklearn.__version__,numpy_version=np.__version__,python_version=sys.version,train_normal=566,train_gas=17,shortshot_available=False,evaluation='full-refit inference, not independent validation',threshold_source='labeled block-excluded normal calibration only')
 path.with_suffix('.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
 return meta
def load_bundle(path):
 path=Path(path);meta=json.loads(path.with_suffix('.json').read_text(encoding='utf-8'))
 if meta['sha256']!=hashlib.sha256(path.read_bytes()).hexdigest():raise ValueError('Model SHA256 mismatch')
 if meta['sklearn_version']!=sklearn.__version__:raise ValueError('Use matching scikit-learn version '+meta['sklearn_version'])
 # joblib is executable pickle; load ONLY this trusted locally generated repository artifact.
 bundle=joblib.load(path)
 if bundle.get('format')!='RG3_GAS_FINAL_FIT_V1':raise ValueError('Unsupported model bundle')
 return bundle


In [ ]:
root=Path.cwd()
if not (root/'data').exists():root=root.parent
d=pd.read_csv(root/'reports/tables/original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.eq('가스'))].reset_index(drop=True)
cuttable=pd.read_csv(root/'reports/tables/rg3_unlabeled_gas_frozen_thresholds.csv')
cuts=dict(zip(cuttable.branch,cuttable.threshold))
bundle=fit_gas_bundle(d,cuts)
meta=save_bundle(bundle,root/'models/rg3_gas_final_fit.joblib')
print(json.dumps(meta,ensure_ascii=False,indent=2))
print('Inference artifact exported. Original labeled evaluation performance is NOT re-evaluated on training rows.')


In [ ]:
root=Path.cwd()
if not (root/'data').exists():root=root.parent
bundle=load_bundle(root/'models/rg3_gas_final_fit.joblib')
old=pd.read_csv(root/'reports/tables/rg3_unlabeled_gas_candidate_predictions.csv',parse_dates=['TimeStamp'])
chosen=old[old.gas_candidate.eq(1)&old.TimeStamp.dt.strftime('%Y-%m-%d').eq('2020-10-20')].TimeStamp.head(5)
parts=[]
for product in ['cn7','rg3']:
 for prefix in ['moldset_labeled_','moldset_unlabeled_']:
  z=pd.read_csv(root/('data/original_split/'+prefix+product+'.csv'));z['product']=product.upper()
  parts.append(normalize_raw(z))
raw=pd.concat(parts,ignore_index=True)
fixture=raw[raw.EQUIP_CD.eq('S14')&raw['product'].eq('RG3')&raw.TimeStamp.isin(chosen)].drop_duplicates(KEYS)
context=raw[raw.EQUIP_CD.eq('S14')&raw.TimeStamp.between(chosen.min()-pd.Timedelta(hours=1),chosen.max())].drop_duplicates(KEYS)
features=prepare_features(fixture,context);expected=predict_cascade(bundle,features)
assert len(expected)==5 and expected.gas_prediction.eq(1).all()
folder=root/'examples';folder.mkdir(exist_ok=True)
fixture.to_csv(folder/'rg3_gas_raw_input.csv',index=False,encoding='utf-8-sig')
context.to_csv(folder/'rg3_gas_raw_history.csv',index=False,encoding='utf-8-sig')
expected.to_csv(folder/'rg3_gas_expected_output.csv',index=False,encoding='utf-8-sig')
print('Inference example:',len(fixture),'queries,',len(context),'history rows; all predictions verified.')
